In [ ]:
# ! pip install google-cloud-aiplatform vertexai

In [1]:
from typing import Sequence
import dataclasses
import json
from vertexai import generative_models

GENERATION_AUTORATER_EVAL_PROMPT_WITH_IMAGES = """
You are an evaluation expert. You will be provided with a query, a generated response and a ground truth. Your task is to analyze the generated response for accuracy, completeness, and relevance compared to the ground truth.
You may also be given with images that provided by generated response or ground truth, in this case, you also need to evaluate if the generated response images are relevant to the ground truth results.

Please provide:
A float score from 0 to 5, where 0 is completely inaccurate and 5 is perfectly accurate.
A detailed explanation of why you gave that score, highlighting specific areas where the generated response was correct, incorrect, missing information, or irrelevant.
The output should be in JSON format.

EXAMPLE:

Query: What's the Capital of China
Generated Response: Victoria.
Ground Truth: Beijing

EVALUATION
{
"score" : 0.0,
"reason" : "The generated response is incorrect based on provided ground truth."
}
"""

RESPONSE_SCHEMA_SCORE = {
    "type": "object",
    "properties": {
        "score": {"type": "number", "minimum": 0, "maximum": 5},
        "reason": {"type": "string"},
    },
    "required": ["score", "reason"],
}


@dataclasses.dataclass(frozen=True)
class GenerationEvaluation:
  """The generation evaluation result.

  Attributes:
    score: The score of the generation evaluation.
    reason: The reason of the generation evaluation.
  """
  score: float
  reason: str


def eval_generation(
    autorater_model: generative_models.GenerativeModel,
    question: Sequence[generative_models.Part],
    model_reply: Sequence[generative_models.Part],
    ground_truth: Sequence[generative_models.Part],
) -> GenerationEvaluation:
  """Evaluates the generated response for accuracy, completeness, and relevance compared to the ground truth.

  Args:
    autorater_model: The autorater model to use.
    question: The question to ask the model.
    model_reply: The model's reply to the question.
    ground_truth: The ground truth answer to the question.

  Returns:
    The generation evaluation result.
  """
  part_list = []
  part_list.append(
      generative_models.Part.from_text(
          GENERATION_AUTORATER_EVAL_PROMPT_WITH_IMAGES
      )
  )

  part_list.append(generative_models.Part.from_text("\nQuery: "))
  for part in question:
    part_list.append(part)

  part_list.append(generative_models.Part.from_text("\nGenerated Response: "))
  for part in model_reply:
    part_list.append(part)

  part_list.append(generative_models.Part.from_text("\nGround Truth: "))
  for part in ground_truth:
    part_list.append(part)

  response: generative_models.GenerationResponse = (
      autorater_model.generate_content(
          part_list,
          generation_config=generative_models.GenerationConfig(
              temperature=0,
              response_mime_type="application/json",
              response_schema=RESPONSE_SCHEMA_SCORE,
          ),
      )
  )
  score_eval_response_dict = json.loads(response.candidates[0].text)
  eval_result = GenerationEvaluation(
      score=score_eval_response_dict["score"],
      reason=score_eval_response_dict["reason"],
  )
  return eval_result

In [2]:
import asyncio

model_name = "gemini-2.5-flash"
model: generative_models.GenerativeModel = generative_models.GenerativeModel(
    model_name=model_name,
)

question = "What is the capital of Canada"
model_reply = "Obviously, Toronto"
ground_truth = "Ottawa"

async def eval_generation_async(question: str, model_reply: str, ground_truth: str, semaphore: asyncio.Semaphore) -> GenerationEvaluation:
    async with semaphore:
        return await asyncio.to_thread(eval_generation,
            autorater_model=model,
            question=[generative_models.Part.from_text(question)],
            model_reply=[generative_models.Part.from_text(model_reply)],
            ground_truth=[generative_models.Part.from_text(ground_truth)]
        )

In [3]:
# corpus = [
#     # 1
#     ("What is the capital of Canada?", "Obviously, Toronto.", "Ottawa"),

#     # 2
#     ("Which planet is closest to the Sun?", "That would be Venus, the hottest planet.", "Mercury"),

#     # 3
#     ("Who wrote the novel '1984'?", "I'm pretty sure that was Aldous Huxley.", "George Orwell"),

#     # 4
#     ("What is the tallest mountain in the world?", "Mount Everest, without a doubt.", "Mount Everest"),

#     # 5
#     ("Who painted the Mona Lisa?", "The famous Michelangelo, of course.", "Leonardo da Vinci"),

#     # 6
#     ("What is the largest mammal on Earth?", "It's the African Elephant.", "Blue Whale"),

#     # 7
#     ("What is the chemical symbol for gold?", "Easy, that's Ag.", "Au"),

#     # 8
#     ("In which city is the Golden Gate Bridge located?", "That's in Los Angeles.", "San Francisco"),

#     # 9
#     ("What is the longest river in the world?", "It has always been the Nile River.", "The Amazon River"),

#     # 10
#     ("Who was the first President of the United States?", "Thomas Jefferson, who wrote the Declaration of Independence.", "George Washington")
# ]

In [4]:
# semaphore = asyncio.Semaphore(10)
# results = await asyncio.gather(*[eval_generation_async(question=question, model_reply=model_reply, ground_truth=ground_truth, semaphore=semaphore) 
#                                 for (question, model_reply, ground_truth) in corpus]
#                                 )

In [5]:
# results

In [6]:
import pandas as pd

# Load claims from cache
df = pd.read_csv("critic_output.csv")

In [7]:
df.head()

,claim,revised_claim,context,is_supported,verdict
0,Blue Ridge Outfitters' flagship stores are loc...,Blue Ridge Outfitters' flagship stores are loc...,Blue Ridge Outfitters' flagship stores in Denv...,False,Accurate
1,The flagship stores for Blue Ridge Outfitters ...,The flagship stores for Blue Ridge Outfitters ...,Blue Ridge Outfitters' flagship stores in Denv...,False,Accurate
2,"The strategy behind the location, design, and ...","The strategy behind the location, design, and ...","The strategy behind their location, design, an...",False,Inaccurate
3,The placement of each flagship store was a ran...,The placement of each flagship store was a del...,The placement of each flagship store was a del...,False,Inaccurate
4,The selection of flagship store locations was ...,This claim is inaccurate. The selection of fla...,The placement of each flagship store was a del...,False,Inaccurate


In [8]:
claims_cached = df.to_dict(orient='records')
claims_cached[0]

{'claim': "Blue Ridge Outfitters' flagship stores are located in Atlanta, Miami, and Boston.",
 'revised_claim': "Blue Ridge Outfitters' flagship stores are located in Denver, Colorado; Bend, Oregon; and Jackson, Wyoming.\n",
 'context': "Blue Ridge Outfitters' flagship stores in Denver, Bend, and Jackson represent more than just retail spaces; they are carefully crafted destinations designed to embody the brand's ethos, serve specific regional communities, and offer unique customer experiences.",
 'is_supported': False,
 'verdict': 'Accurate'}

# Claims need a corresponding query


In [9]:
import dspy
import mlflow
mlflow.dspy.autolog()
mlflow.set_experiment("generate_queries")

model_name = "gemini/gemini-2.5-pro-preview-06-05"
# model_name = "gemini/gemini-2.0-flash"
lm = dspy.LM(
    model=model_name,
    max_tokens=65535,
    # allowed_openai_params=["thinking"],
    # thinking={"type": "enabled", "budget_tokens": 1024},
)
dspy.configure(lm=lm)

/Users/ivanmkc/code/adk-samples/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# from typing import Literal

# class QueryGenerationSignature(dspy.Signature):
#     """
#     Given a claim and its original context, generate a question where the claim is the direct answer.
#     The style of the question is determined by the ambiguity_level.

#     - If 'straightforward', the question should be direct and factual, often starting with Who, What, When, or Where. The claim should be the most obvious and concise answer.
#     - If 'ambiguous', the question should be more open-ended, subjective, or analytical, perhaps starting with Why, How, or asking for significance/implications. The claim should still be a valid and strong answer, but other interpretations might be possible.
#     - Do not use questions like "What would be an inaccurate way to ..." or "What is a false statement regarding..."
#     """

#     context: str = dspy.InputField(
#         desc="The source text where the claim originates, providing necessary context."
#     )
#     claim: str = dspy.InputField(
#         desc="The specific statement that should be the answer to the generated question."
#     )
#     ambiguity_level: str = dspy.InputField(
#         desc="Controls the style of the generated query. Options: 'straightforward' or 'ambiguous'."
#     )
#     query: str = dspy.OutputField(
#         desc="The generated question for which the claim is the intended answer."
#     )


# class QueryGenerator(dspy.Module):
#     """A module to generate a query from a claim and context."""
#     def __init__(self):
#         super().__init__()
#         self.query_generator = dspy.ChainOfThought(QueryGenerationSignature)

#     async def forward(self, claim: str, context: str, ambiguity: Literal["straightforward", "ambiguous"]) -> dspy.Prediction:
#         """
#         Generates a query from a claim and its context.

#         Args:
#             claim: The statement that should be the answer to the query.
#             context: The original text providing context for the claim.
#             ambiguity: The desired ambiguity level of the query.
#                       Must be either 'straightforward' or 'more ambiguous'.

#         Returns:
#             A dspy.Prediction object containing the generated 'query'.
#         """
#         if ambiguity not in ["straightforward", "ambiguous"]:
#             raise ValueError("ambiguity must be 'straightforward' or 'ambiguous'")
            
#         query_generator_async = dspy.asyncify(self.query_generator)

#         prediction = await query_generator_async(
#             claim=claim,
#             context=context,
#             ambiguity_level=ambiguity
#         )
#         return prediction

In [19]:
# ! pip install async-lru --quiet

In [ ]:
# from tqdm.asyncio import tqdm
# from async_lru import alru_cache

# query_generator_module = QueryGenerator()

# semaphore = asyncio.Semaphore(10)

# @alru_cache(maxsize=None)
# async def generate_query(claim: str, context: str, ambiguity: str) -> str:
#     async with semaphore:
#         query = await query_generator_module.forward(
#             claim=claim,
#             context=context,
#             ambiguity=ambiguity
#         )

#         return query

# ambiguity = "straightforward"
# queries = await tqdm.gather(*[
#         generate_query(
#             claim=claim["claim"], 
#             context=claim["context"], 
#             ambiguity=ambiguity) 
#             for claim in claims_cached[:10]
#         ]
#     )



01:06:56 - LiteLLM:INFO: utils.py:2826 - 
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
01:06:56 - LiteLLM:INFO: utils.py:2826 - 
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
01:06:56 - LiteLLM:INFO: utils.py:2826 - 
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
01:06:56 - LiteLLM:INFO: utils.py:2826 - 
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
01:06:56 - LiteLLM:INFO: utils.py:2826 - 
LiteLLM completion() model= gemini-2.5-pro-preview-06-05; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-

In [21]:
queries

[Prediction(
     reasoning='The user wants a straightforward question for which the provided claim is the answer. The claim is a list of cities where Blue Ridge Outfitters\' flagship stores are located. Therefore, the most direct and straightforward question is to ask for the locations of these stores using a "Where" question.',
     query="Where are Blue Ridge Outfitters' flagship stores located?"
 ),
 Prediction(
     reasoning='The user wants a straightforward question where the claim is the answer. The provided claim is a direct contradiction of the information in the context. The context states that the stores are "more than just retail spaces" and are "carefully crafted destinations," while the claim asserts they are "nothing more than simple retail spaces." To make the incorrect claim the correct answer to a straightforward question, the question must ask for a mistaken or inaccurate viewpoint. By asking for a "misconception," the query directly prompts for the provided claim, 

In [ ]:
semaphore = asyncio.Semaphore(10)
results = await asyncio.gather(*[eval_generation_async(
    question=claim_info, 
    model_reply=model_reply, 
    ground_truth=ground_truth, 
    semaphore=semaphore) 
    for claim_info in claims_cached]
    )

In [ ]:
import pandas as pd

pd.DataFrame([dict(question=question, 
                   model_reply=model_reply,
                   ground_truth=ground_truth,
                   score=result.score,
                   reason=result.reason) 
              for ((question, model_reply, ground_truth), result) in zip(corpus, results)])